In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
Q= torch.tensor([[2.,3.,8.,9.,0.],[1.,2.,8.,7.,1.],[9.,7.,3.,2.,9.]])
K=torch.tensor([[3.,6.,8.,9.,1.],[7.,3.,4.,1.,0.],[2.,8.,9.,1.,3.]])
V=torch.tensor([[4.,6.,8.,9.,1.],[7.,8.,9.,1.,7.],[2.,1.,3.,4.,8.]])

S=Q @ K.T
d= torch.softmax(S,dim=-1)
F= d @ V
print(F)
print(d)
print(S)

tensor([[4.0000, 6.0000, 8.0000, 9.0000, 1.0000],
        [4.0000, 6.0000, 8.0000, 9.0000, 1.0000],
        [2.0001, 1.0002, 3.0002, 4.0002, 7.9997]])
tensor([[1.0000e+00, 0.0000e+00, 8.7565e-27],
        [1.0000e+00, 3.0144e-40, 2.1151e-19],
        [4.5398e-05, 1.2664e-14, 9.9995e-01]])
tensor([[169.,  64., 109.],
        [143.,  52., 100.],
        [120.,  98., 130.]])


In [3]:
import torch
torch.manual_seed(0)
def attention(W_q,W_k,W_v,X):
    Q=X @ W_q
    K= X @ W_k
    V= X @ W_v
    score= Q @ K.T/K.shape[-1]**0.5
    weights = torch.softmax(score,dim=-1)
    out = weights @ V
    return score , weights, out, Q, K, V
W_q= torch.randn(512,5)/512**0.5
W_k=torch.randn(512,5)/512**0.5
W_v=torch.randn(512,5)/512**0.5
X=torch.randn(3,512)
score,weights,out,Q,K,V= attention(W_q,W_k,W_v,X)
print(score, weights,out,Q,K,V)
print(score.shape,weights.shape,out.shape)

print("\n\nseparate from above code , only for verification ")

manual= weights[0,0]*V[0] + weights[0,1]*V[1] + weights[0,2]*V[2]
print(manual)
print(out[0])

tensor([[ 0.2609, -0.5119, -0.1389],
        [-0.4669,  0.0088, -0.0457],
        [-0.6698, -1.9923,  0.4939]]) tensor([[0.4690, 0.2166, 0.3144],
        [0.2420, 0.3893, 0.3687],
        [0.2238, 0.0596, 0.7165]]) tensor([[-0.7092, -0.5292, -0.0450,  0.1246, -1.2361],
        [-0.8767, -0.4309, -0.3256, -0.0284, -0.9286],
        [-0.6828, -1.1100, -0.4563,  0.6516, -1.9347]]) tensor([[ 0.3410,  0.3670,  1.3952,  0.5903, -0.5516],
        [-1.5227,  0.2577,  0.1818, -0.6485, -0.9282],
        [ 0.6211, -0.5582, -0.4827,  0.9731, -1.3432]]) tensor([[ 0.4750,  1.0462,  0.3413, -0.0525,  0.7392],
        [-1.1366,  1.7724, -0.1870,  0.1245,  2.2118],
        [ 0.3836, -0.5095, -0.2775, -0.2682, -0.5276]]) tensor([[-0.4305, -0.3839,  0.6705,  0.1044, -1.3927],
        [-1.3126,  0.5268, -0.4903, -1.0508,  0.6790],
        [-0.7092, -1.4730, -0.8055,  0.9642, -2.3216]])
torch.Size([3, 3]) torch.Size([3, 3]) torch.Size([3, 5])


separate from above code , only for verification 
tensor([-0.7

In [4]:
import torch
torch.manual_seed(0)
def attention(W_q,W_k,W_v,X):
    Q= X @ W_q
    K= X @ W_k
    V= X @ W_v
    score= Q @ K.T/K.shape[-1]**0.5
    weights= torch.softmax(score, dim=-1)
    out = weights @ V
    return Q, K,V, score, weights, out

##setup----
d_model= 512
num_heads= 8 
d_head= d_model// num_heads

X= torch.randn(3,d_model)

##setting up weights for multi heads-----
W_qs= [torch.randn(d_model, d_head)/d_model**0.5 for _ in range(num_heads)]
W_ks= [torch.randn(d_model, d_head)/d_model**0.5 for _ in range(num_heads)]
W_vs= [torch.randn(d_model, d_head)/d_model**0.5 for _ in range(num_heads)]

##collecting head weights and outputs in a list-----
head_weights=[]
head_out= []
for h in range(num_heads):
    Q,K,V,score,weights,out= attention(W_qs[h],W_ks[h],W_vs[h],X)
    head_weights.append(weights)
    head_out.append(out)

concat=torch.concat(head_out,dim=-1)
W_o= torch.randn(d_model,d_model)/d_model**0.5

final= concat @ W_o

##print("query,key, value, score, weights, X in order:",Q,K,V,score, weights,X)
##print("\n\nfinalised output after multi head attention:",final)
print(len(head_out), head_out[0].shape)   # 8, [3, 64]
print(concat.shape)                               # [3, 512]
print(final.shape)

# 1. shapes first (head 7's values)
print(Q.shape, K.shape, V.shape)
print(score.shape, weights.shape, out.shape)

# 2. every row of weights must add to 1
print(weights.sum(dim=-1))

# 3. hand-check one score: word 0 looking at word 1
print(score[0, 1])
print((Q[0] @ K[1]) / 64**0.5)          # same number

# 4. hand-check one output row (a blend of V rows)
manual = weights[0,0]*V[0] + weights[0,1]*V[1] + weights[0,2]*V[2]
print(torch.allclose(manual, out[0], atol=1e-5))   # True

# 5. confirm the printout was head 7
print(torch.equal(weights, head_weights[7]))       # True

# 6. compare two heads
print(head_weights[0])
print(head_weights[1])

# 7. stapling check: head 3's notes sit in columns 192 to 255
print(torch.equal(concat[:, 3*64:4*64], head_out[3]))   # True

# 8. peek into big tensors
print(final[0, :5])                  # word 0, first 5 numbers
torch.set_printoptions(sci_mode=False, precision=4)
print(K)                             # now in normal decimals

8 torch.Size([3, 64])
torch.Size([3, 512])
torch.Size([3, 512])
torch.Size([3, 64]) torch.Size([3, 64]) torch.Size([3, 64])
torch.Size([3, 3]) torch.Size([3, 3]) torch.Size([3, 64])
tensor([1.0000, 1.0000, 1.0000])
tensor(1.4804)
tensor(1.4804)
True
True
tensor([[0.1685, 0.4169, 0.4146],
        [0.0497, 0.4942, 0.4561],
        [0.5304, 0.0242, 0.4454]])
tensor([[0.5193, 0.4158, 0.0649],
        [0.2793, 0.4709, 0.2498],
        [0.7212, 0.2076, 0.0712]])
True
tensor([-0.4171, -0.5321,  0.2669, -1.1797, -0.8802])
tensor([[-1.8837, -0.2153, -1.2504,  0.6034, -0.2005, -1.3808, -2.4033,  1.1002,
         -0.3253,  0.1554, -0.2366,  0.8369, -0.7167, -1.7659, -0.6512,  0.2400,
          0.5251,  0.6958,  0.5341, -2.8184, -0.4869, -0.1707,  0.2568, -1.8761,
          1.8120,  0.0765,  1.5204, -0.7580, -1.0735,  0.8765,  2.1882, -1.3688,
         -0.4316, -0.7911,  0.4899, -0.6200, -0.3792,  0.3110,  0.3275,  0.2615,
         -0.9415,  1.6005,  0.6381,  0.2188, -0.1578,  0.6713,  0.4991,  0.

In [5]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length=5000, dropout=0.1):
        super().__init__()
        self.dropout= nn.Dropout(dropout)
        lam= torch.exp(torch.arange(0,d_model,2) * (-math.log(10000)/d_model))
        position= torch.arange(max_length).unsqueeze(1)
        pe= torch.zeros(max_length, d_model)
        pe[:,0::2]= torch.sin(lam * position)
        pe[:,1::2]= torch.cos(lam * position)
        self.register_buffer("pe",pe.unsqueeze(0))

    def forward(self,x):
        x= x + self.pe[:,: x.size(1)]
        return self.dropout(x)

In [6]:
d_model, max_len = 512, 100
pe = PositionalEncoding(d_model, max_len, dropout=0.0).pe[0]   # (max_len, d_model)

# Test 1: one entry against the paper's formula
pos, i = 7, 3
ang = torch.tensor(pos / 10000 ** (2 * i / d_model))
print(torch.isclose(pe[pos, 2*i],   torch.sin(ang), atol=1e-5))
print(torch.isclose(pe[pos, 2*i+1], torch.cos(ang), atol=1e-5))

# Test 2: the shift recipe, a*s + b*c, on every pair at once
k = 5
lam = 10000 ** (-torch.arange(0, d_model, 2) / d_model)
a, b = torch.cos(lam * k), torch.sin(lam * k)
s, c = pe[:-k, 0::2], pe[:-k, 1::2]                  # heights at t
new_s, new_c = a*s + b*c, a*c - b*s                  # recipe
print(torch.allclose(new_s, pe[k:, 0::2], atol=1e-4))
print(torch.allclose(new_c, pe[k:, 1::2], atol=1e-4))

tensor(True)
tensor(True)
True
True


In [7]:
import  torch
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, d_model= 512, d_ff= 2048):
        super().__init__()
        self.expand= nn.Linear(d_model, d_ff)
        self.squeeze= nn.Linear(d_ff, d_model)

    def forward(self,x):
        x= self.expand(x)
        x= torch.relu(x)
        x= self.squeeze(x)
        return x


#Test
ffn= FeedForward()
x= torch.randn(3,512)
out= ffn(x)
print("shape of x\n",x.shape, "shape of out\n",out.shape)

shape of x
 torch.Size([3, 512]) shape of out
 torch.Size([3, 512])


In [8]:
#Embeddings code 
import torch
import torch.nn as nn
import math

class Tokenembedding(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.d_model = d_model
        self.table= nn.Embedding(vocab_size, d_model)
    def forward(self, ids):
        return self.table(ids) * math.sqrt(self.d_model)

torch.manual_seed(0)
emb= Tokenembedding(512, 37000)
ids= torch.tensor([5, 912, 4])
out = emb(ids)
print(out.shape)
print(out)

torch.Size([3, 512])
tensor([[  4.0736,   3.4174, -26.1055,  ..., -14.4365, -26.5052, -19.0421],
        [  6.3134,  16.5026,  13.5603,  ..., -16.4548, -12.8007,  -7.8124],
        [ 15.0630,  20.0184,  10.5695,  ...,  13.0250,  -0.7732,  31.6952]],
       grad_fn=<MulBackward0>)
